# Activation Patching

TODO: Add desc

## Inital Setup

In [39]:
# For loading HF_Token
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [40]:
# Import stuff
import torch
import torch.nn as nn
import einops
from fancy_einsum import einsum
import tqdm.auto as tqdm
import plotly.express as px

from jaxtyping import Float
from functools import partial

In [41]:
# Setting plotly renderer to work for notebooks
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [42]:
import circuitsvis as cv

# Testing that the library works
cv.examples.hello("Fin")

In [43]:
# import transformer_lens
import transformer_lens.utilities as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import FactoredMatrix, HookedTransformer
from transformer_lens.model_bridge import TransformerBridge

In [44]:
# Turn off auto differentation to save memory
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [45]:
def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": xaxis, "y": yaxis},
        **kwargs,
    ).show(renderer)


def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.line(y=utils.to_numpy(tensor), labels={"x": xaxis, "y": yaxis}, **kwargs).show(renderer)


def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
    x = utils.to_numpy(x)
    y = utils.to_numpy(y)
    px.scatter(y=y, x=x, labels={"x": xaxis, "y": yaxis, "color": caxis}, **kwargs).show(renderer)

## Activation Patching Code

In [46]:
device = utils.get_device()

In [47]:
model = TransformerBridge.boot_transformers("openai-community/gpt2", device=device)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7824.11it/s]


In [ ]:
def logits_to_logit_diff(logits, correct_answer=" John", incorrect_answer=" Mary"):
    correct_index = model.to_single_token(correct_answer)
    incorrect_index = model.to_single_token(incorrect_answer)
    return logits[0, -1, correct_index] - logits[0, -1, incorrect_index]

In [68]:
clean_input = "Jack and Jill went on a date, Jack gave flowers to"
corrupted_input = "Jack and Jill went on a date, Jill gave flowers to"

clean_tokens = model.to_tokens(clean_input)
clean_logits, clean_cache = model.run_with_cache(clean_input)
clean_logit_diff = logits_to_logit_diff(clean_logits)


corrupted_tokens = model.to_tokens(corrupted_input)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_input)
corrupted_logit_diff = logits_to_logit_diff(corrupted_logits)

print(clean_logit_diff)
print(corrupted_logit_diff)


tensor(-0.7826)
tensor(1.3313)


Now we must do the 'patching', we have to get the activations from the corrupted input and patch them into the clean input and then see how the final logit differs.

for every atten head
    replace clean with corrupted atten

In [70]:
from functools import partial
from transformer_lens import utils

def activation_patching_hook(value, hook, head=None):
    clean_head = clean_cache[hook.name]
    value[:, :, head, :] = clean_head[:, :, head, :]
    return value

results = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)

for layer in tqdm.tqdm(range(model.cfg.n_layers)):
    for head in range(model.cfg.n_heads):
        hook_fn = partial(activation_patching_hook, head=head)
        patched_logits = model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[(utils.get_act_name("z", layer), hook_fn)]
        )
        
        patched_logit_diff = logits_to_logit_diff(patched_logits)
        results[layer, head] = (patched_logit_diff - corrupted_logit_diff) / (
            clean_logit_diff - corrupted_logit_diff
        )
        

100%|██████████| 12/12 [00:08<00:00,  1.43it/s]


In [71]:
imshow(
    results,
    xaxis="Head",
    yaxis="Layer",
    title="Activation Patching: Loss by Layer and Head",
)